In [2]:
import os
import json
import pandas as pd

test_df = pd.DataFrame(json.load(open("/home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/extract_F_only_6ch/test.json")))
tr_df = pd.DataFrame(json.load(open("/home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/extract_F_only_6ch/train.json")))
val_df = pd.DataFrame(json.load(open("/home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/extract_F_only_6ch/val.json")))

In [16]:
merged_df = pd.concat([tr_df, val_df, test_df], axis=0)
len(merged_df)

4825

In [14]:
i=0
forehead_data = []

for root, dirs, files in os.walk('/home/work/hocheol_dir/workspace/data/028_data/028_forehead'):
    for file in files:
        if file.lower().endswith('.jpg'):
            origin_forehead_path = os.path.join(root, file)
            id = origin_forehead_path.split('/')[-2]
            forehead_data.append({'id':id, 'origin_forehead_path':origin_forehead_path})

In [31]:
forehead_df = pd.DataFrame(forehead_data).sort_values(by=['id', 'origin_forehead_path']).reset_index(drop=True)
forehead_df.head()

,id,origin_forehead_path
0,0001,/home/work/hocheol_dir/workspace/data/028_data...
1,0001,/home/work/hocheol_dir/workspace/data/028_data...
2,0001,/home/work/hocheol_dir/workspace/data/028_data...
3,0001,/home/work/hocheol_dir/workspace/data/028_data...
4,0001,/home/work/hocheol_dir/workspace/data/028_data...


In [32]:
new_tr_df = pd.merge(tr_df, forehead_df, on='id', how='left')
new_val_df = pd.merge(val_df, forehead_df, on='id', how='left')
new_test_df = pd.merge(test_df, forehead_df, on='id', how='left')

In [39]:
################################# 이걸 바로 json으로 만들면 안됨. F, Fb, Ft의 매핑이 제대로 안되어있음
new_tr_df['new_forehead_path'] = new_tr_df.apply(
    lambda row: os.path.join(os.path.dirname(row['left_cheek_path']),
                             os.path.basename(row['origin_forehead_path'])),
    axis=1
)

new_val_df['new_forehead_path'] = new_val_df.apply(
    lambda row: os.path.join(os.path.dirname(row['left_cheek_path']),
                             os.path.basename(row['origin_forehead_path'])),
    axis=1
)

new_test_df['new_forehead_path'] = new_test_df.apply(
    lambda row: os.path.join(os.path.dirname(row['left_cheek_path']),
                             os.path.basename(row['origin_forehead_path'])),
    axis=1
)


In [54]:
for row in new_tr_df.iterrows():
    print(row)
    break

(0, id                                                                   0001
left_cheek_path                 Training/0001/left_cheek_01_0001_01_F.jpg
right_cheek_path               Training/0001/right_cheek_01_0001_01_F.jpg
pigment_count                                                         147
label                                                                   2
origin_forehead_path    /home/work/hocheol_dir/workspace/data/028_data...
new_forehead_path                    Training/0001/forehead_0001_01_F.jpg
Name: 0, dtype: object)


In [56]:
import shutil
from tqdm import tqdm

root_dir = '/home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp'

for row in tqdm(new_tr_df.iterrows(), desc='train'):
    shutil.copy(row[1]['origin_forehead_path'], os.path.join(root_dir, row[1]['new_forehead_path']))

for row in tqdm(new_val_df.iterrows(), desc='val'):
    shutil.copy(row[1]['origin_forehead_path'], os.path.join(root_dir, row[1]['new_forehead_path']))

for row in tqdm(new_test_df.iterrows(), desc='test'):
    shutil.copy(row[1]['origin_forehead_path'], os.path.join(root_dir, row[1]['new_forehead_path']))

train: 19300it [01:03, 301.65it/s]
val: 2400it [00:08, 298.25it/s]
test: 2425it [00:07, 303.74it/s]


In [58]:
new_tr_df.drop(columns=['origin_forehead_path', 'new_forehead_path'], inplace=True)
new_val_df.drop(columns=['origin_forehead_path', 'new_forehead_path'], inplace=True)
new_test_df.drop(columns=['origin_forehead_path', 'new_forehead_path'], inplace=True)

In [60]:
def make_forehead_path(row):
    left_cheek_path = row['left_cheek_path']
    forehead_path = left_cheek_path.replace('left_cheek', 'forehead')
    return forehead_path

In [65]:
tr_df['forehead_path'] = tr_df.apply(lambda row: make_forehead_path(row), axis=1)
val_df['forehead_path'] = val_df.apply(lambda row: make_forehead_path(row), axis=1)
test_df['forehead_path'] = test_df.apply(lambda row: make_forehead_path(row), axis=1)

In [89]:
base_dirs = ['/home/work/hocheol_dir/workspace/data/annotations/028_data/Training/02.라벨링데이터/1. 디지털카메라', '/home/work/hocheol_dir/workspace/data/annotations/028_data/Validation/02.라벨링데이터/1. 디지털카메라']

json_data = []

for base_dir in base_dirs:
    for root, dirs, files in os.walk(base_dir):
        json_files = [f for f in files if f.lower().endswith('.json')]
        if json_files:
            with open(os.path.join(root, json_files[0]), 'r') as f:
                json_data.append(json.load(f))

len(json_data)

965

In [90]:
ids = set()
for data in json_data:
    ids.add(data['info']['id'])
len(ids)

965

In [97]:
sum([len(tr_df['id'].unique()), len(val_df['id'].unique()), len(test_df['id'].unique())])

965

In [100]:
age_list = []
for data in json_data:
    id = data['info']['id']
    age = data['info']['age']
    age_list.append({'id':id, 'age':age})

In [101]:
len(age_list)

965

In [109]:
age_df = pd.DataFrame(age_list)
age_df.sort_values(by=['id'], inplace=True)
age_df.reset_index(drop=True, inplace=True)
age_df

,id,age
0,0001,55
1,0002,50
2,0003,24
3,0004,47
4,0006,55
...,...,...
960,1096,25
961,1097,24
962,1098,23
963,1099,26


In [111]:
tr_df = pd.merge(tr_df, age_df, on='id', how='left')
val_df = pd.merge(val_df, age_df, on='id', how='left')
test_df = pd.merge(test_df, age_df, on='id', how='left')
tr_df['age'] = tr_df['age'].astype(int)
val_df['age'] = val_df['age'].astype(int)
test_df['age'] = test_df['age'].astype(int)

In [121]:
with open('/home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/F_only_with_age/train.json', 'w') as f:
    json.dump(tr_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)
with open('/home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/F_only_with_age/val.json', 'w') as f:
    json.dump(val_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)
with open('/home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/F_only_with_age/test.json', 'w') as f:
    json.dump(test_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

In [ ]:
import os

################### forehead path 다른 파일이랑 규칙 달라서 고쳐줌

base_dir = '/home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp'

for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.startswith('forehead_') and file.lower().endswith('.jpg'):
            parts = file.split('_')
            
            if len(parts) >= 4:
                # parts = ['forehead', '0008', '01', 'F.jpg']
                id1 = parts[1]
                id2 = parts[2]
                rest = '_'.join(parts[3:])

                new_filename = f"forehead_{id2}_{id1}_{id2}_{rest}"
                
                old_path = os.path.join(root, file)
                new_path = os.path.join(root, new_filename)

                print(f"Renaming: {old_path} → {new_path}")
                os.rename(old_path, new_path)


Renaming: /home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/Test/0008/forehead_0008_01_F.jpg → /home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/Test/0008/forehead_01_0008_01_F.jpg
Renaming: /home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/Test/0008/forehead_0008_01_Fb.jpg → /home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/Test/0008/forehead_01_0008_01_Fb.jpg
Renaming: /home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/Test/0008/forehead_0008_01_Ft.jpg → /home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/Test/0008/forehead_01_0008_01_Ft.jpg
Renaming: /home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/Test/0008/forehead_0008_02_F.jpg → /home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/Test/0008/forehead_02_0008_02_F.jpg
Renaming: /home/work/hocheol_dir/workspace/data/028_data/028_cropped_data_mp/Test/0008/forehead_0008_03_F.jpg → /home/work/hocheol_dir/w

In [142]:
merged_df = pd.concat([tr_df, val_df, test_df], axis=0)
for row in merged_df.iterrows():
    if not (os.path.exists(os.path.join(root_dir, row[1]['left_cheek_path'])) and os.path.exists(os.path.join(root_dir, row[1]['forehead_path']))):
        print(f"Missing file: {row[1]['left_cheek_path']}, {row[1]['forehead_path']}, {row[1]['new_forehead_path']}")